In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

import statsmodels.api as sm  # Importa statsmodels
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mean_squared_error

from sklearn.preprocessing import OneHotEncoder
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.ensemble import RandomForestRegressor

from xgboost import XGBRegressor

In [ ]:
# Donde se coloque el archivo con las salidas correspondientes (cambiar según preprocesamiento):
# df = pd.read_csv(r'C:\output_training_FEF.csv', delimiter=';', encoding='latin1')
# df = pd.read_csv(r'C:\output_training_FFEF.csv', delimiter=';', encoding='latin1')

In [ ]:
df_ml = df.copy()
df_ml

In [ ]:
# Seleccionamos solo las variables categóricas
cat = df_ml.select_dtypes(include='object')  # o 'category' si usas categorías

# Instanciamos OneHotEncoder
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')  # 'sparse_output' es usado en versiones recientes

# Entrenamos el codificador
ohe.fit(cat)

# Aplicamos la transformación a las variables categóricas
cat_ohe = ohe.transform(cat)

# Ponemos los nombres a las columnas codificadas
cat_ohe_df = pd.DataFrame(cat_ohe, columns=ohe.get_feature_names_out(input_features=cat.columns)).reset_index(drop=True)

# Concatenamos con las demás variables del dataset original si es necesario
df_ml_final = pd.concat([df_ml.reset_index(drop=True), cat_ohe_df], axis=1).drop(columns=cat.columns)

In [ ]:
# Seleccionamos las variables numéricas para poder juntarlas a las cat_hoe
num = df.Improve

In [ ]:
# Las juntamos todas en el dataframe final
df_ml = pd.concat([cat_ohe_df,num,df.Alfa], axis = 1)
df_ml.info()

In [ ]:
#Separación predictoras y target para IMPROVE
X = df_ml.drop(columns='Improve')
y = df_ml['Improve']

In [ ]:
##################################################
SEED = 123

# Se introduce semilla para reproducibilidad

In [ ]:
# División en conjuntos de entrenamiento y prueba
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)

In [ ]:
##### Nuevas pruebas de Machine learning:
X_train

# **REGRESIÓN LOGÍSTICA**

In [ ]:
# ENTRENAMIENTO DEL MODELO SOBRE TRAIN CON REGRESIÓN LOGÍSTICA

# Creación del modelo utilizando matrices como en scikitlearn
# ==============================================================================
# A la matriz de predictores se le tiene que añadir una columna de 1s para el intercept del modelo
X_train_smlog = sm.add_constant(X_train, prepend=True)
X_test_smlog = sm.add_constant(X_test, prepend=True)

model_sm = sm.Logit(endog=y_train, exog=X_train_smlog,).fit()
print(model_sm.summary())

# Calcula la importancia relativa en porcentaje
total_importance = np.abs(model_sm.params[1:]).sum()
importance_percent = (np.abs(model_sm.params[1:])/ total_importance) * 100

# Imprime la importancia relativa en porcentaje para cada variable
for feature, importance in zip(X.columns, importance_percent):
    print(f"{feature}: {importance:.2f}%")

# Realiza predicciones en el conjunto de prueba
y_pred_smlog = model_sm.predict(X_test_smlog)

# Evalúa el rendimiento del modelo
mse = mean_squared_error(y_test, y_pred_smlog)
print(f'Mean Squared Error: {mse}')

In [ ]:
varianza_explicada_sm = model_sm.prsquared
# varianza explicada total modelo 
print(varianza_explicada_sm)

In [ ]:
# varianza explicada sin alpha tras ejecutar otra vez sin alpha
varianza_explicada_sm_alpha = model_sm.prsquared
print(varianza_explicada_sm_alpha)

# **REDES NEURONALES**

In [ ]:
# ENTRENAMIENTO DEL MODELO SOBRE TRAIN CON REDES NEURONALES

# =====================================================
# Fijar TODAS las semillas
# =====================================================
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

# =====================================================
# Definir modelo
# =====================================================
def build_model(neurons=10, learning_rate=0.001):
    tf.random.set_seed(SEED)  # importante dentro del modelo
    
    model = keras.Sequential([
        # este es con alpha
        #layers.Dense(neurons, activation='relu', input_shape=(7,)), 
        layers.Dense(neurons, activation='relu', input_shape=(6,)), #sin alpha
        layers.Dense(1)
    ])
    
    optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
    model.compile(optimizer=optimizer, loss='mse')
    
    return model

model = KerasRegressor(model=build_model, verbose=0)

# =====================================================
# Grid reducido y controlado
# =====================================================
param_grid = {
    "model__neurons": [5, 10, 20],
    "model__learning_rate": [0.001, 0.01],
    "batch_size": [16, 32],
    "epochs": [50, 100]
}

cv = KFold(n_splits=3, shuffle=True, random_state=SEED)

grid = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=cv,
    scoring="neg_mean_squared_error",
    n_jobs=1   # CLAVE para reproducibilidad
)

# =====================================================
# Entrenamiento
# =====================================================
grid.fit(X_train, y_train)

print("Mejores parámetros:", grid.best_params_)

model_nn = grid.best_estimator_.model_

# =====================================================
# Evaluación final
# =====================================================
y_pred_nn = model_nn.predict(X_test)

mse = mean_squared_error(y_test, y_pred_nn)
r2 = r2_score(y_test, y_pred_nn)

print("MSE:", mse)
print("R²:", r2)

print(f'Mean Squared Error: {mse}')

In [ ]:
# Varianza explicada del modelo
varianza_explicada_nn = r2_score(y_test, y_pred_nn)
print(varianza_explicada_nn)

In [ ]:
# Varianza explicada sin alpha
varianza_explicada_nn_alpha = r2_score(y_test, y_pred_nn)
print(varianza_explicada_nn_alpha)

# **RANDOM FOREST**

In [ ]:
# ENTRENAMIENTO DEL MODELO SOBRE TRAIN CON RANDOM FOREST

# Crear y entrenar el modelo Random Forest
model_rf = RandomForestRegressor(n_estimators=200, random_state=SEED)
model_rf.fit(X_train, y_train)

# Realizar predicciones en el conjunto de prueba
y_pred_rf = model_rf.predict(X_test)


# Calcular el error cuadrático medio
mse = mean_squared_error(y_test, y_pred_rf)
print(f'Mean Squared Error: {mse}')

# Obtener la importancia relativa de las variables en porcentaje
importance_percentages = (model_rf.feature_importances_ / np.sum(model_rf.feature_importances_)) * 100

# Crear un DataFrame para mostrar los resultados
importance_df = pd.DataFrame({
    'Variable': X_train.columns,
    'Importance Percentage': importance_percentages
})


# Mostrar el DataFrame con la importancia relativa
print(f"{importance_df}%")

# Visualizar la importancia relativa en un gráfico de barras
plt.bar(importance_df['Variable'], importance_df['Importance Percentage'])
plt.xlabel('Variable')
plt.ylabel('Importance Percentage')
plt.title('Feature Importance')
plt.show()

In [ ]:
varianza_explicada_rf =r2_score(y_test, y_pred_rf)
# varianza explicada total modelo 
print(varianza_explicada_rf)

In [ ]:
# Varianza explicada sin alpha
varianza_explicada_rf_alpha = r2_score(y_test, y_pred_rf)
print(varianza_explicada_rf_alpha)

# **XGBoost (EXTREME GRADIENT BOOSTING)**

In [ ]:
# Crear y entrenar el modelo XGBoost
model_xgb = XGBRegressor(n_estimators=50, max_depth = 3, random_state=SEED)
model_xgb.fit(X_train, y_train)

# Realizar predicciones en el conjunto de prueba
y_pred_xgb = model_xgb.predict(X_test)

# Calcular el error cuadrático medio
mse = mean_squared_error(y_test, y_pred_xgb)
print(f'Mean Squared Error: {mse}')

# Obtener la importancia relativa de las variables en porcentaje
importance_percentages = (model_xgb.feature_importances_ / np.sum(model_xgb.feature_importances_)) * 100

# Crear un DataFrame para mostrar los resultados
importance_df = pd.DataFrame({
    'Variable': X_train.columns,
    'Importance Percentage': importance_percentages
})

# Mostrar el DataFrame con la importancia relativa
print(importance_df)

In [ ]:
varianza_explicada_xgb =r2_score(y_test, y_pred_xgb)
# varianza explicada total modelo 
print(varianza_explicada_xgb)

In [ ]:
# Varianza explicada sin alpha
varianza_explicada_xgb_alpha = r2_score(y_test, y_pred_xgb)
print(varianza_explicada_xgb_alpha)

# **PREDICCIONES**

## Predicción: Regresión Logística

In [ ]:
# Automatizado.
# Crear un DataFrame con un rango de valores de Alpha de 0 a 1 con paso de 0.01
alpha_values = pd.DataFrame({'Alfa': [i / 100 for i in range(101)]})

# Añadir una columna 'const' con valores 1
alpha_values['const'] = 1

# Algoritmos y colores especificados (usando los nombres que el modelo espera)
algorithms = ['Algoritmo_Louvain', 'Algoritmo_Leiden', 'Algoritmo_Walktrap', 
              'Algoritmo_Infomap', 'Algoritmo_Fast Greedy', 'Algoritmo_Surprise']
colors = ['blue', 'green', 'yellow', 'grey', 'purple', 'pink']

# Listas para guardar las predicciones de cada algoritmo
predictions_dict = {}

# Para cada algoritmo, realizar las predicciones y guardar el gráfico
for idx, algorithm in enumerate(algorithms):
    # Establecer el valor de 1 para el algoritmo actual y 0 para los demás
    # Primero restablecemos todos los algoritmos a 0
    alpha_values[algorithms] = 0
    
    # Asignamos 1 al algoritmo actual
    alpha_values[algorithm] = 1

    # Crear el conjunto de características X_test para este algoritmo
    # Usamos feature_names_in_ para asegurarnos del orden correcto
    #X_test_p = alpha_values[model_sm.feature_names_in_]
    # Usamos esto para Logit y RN:
    X_test_p = alpha_values[model_sm.model.exog_names]
    
    # Realizar las predicciones con el modelo de Regresión Logistica
    y_pred = model_sm.predict(X_test_p)
    
    # Guardar las predicciones
    predictions_dict[algorithm] = y_pred
    
    # Crear un gráfico de dispersión para este algoritmo
    alfas = alpha_values['Alfa']
    predicciones = y_pred

    # Crear gráfico
    plt.figure(figsize=(8, 6))
    plt.scatter(alfas[:-1], predicciones[:-1], label='Predictions', color=colors[idx], marker='o', s=20)  # Excluye la última predicción
    plt.scatter(alfas.iloc[-1], predicciones.iloc[-1], label='Last Prediction', color='red', marker='o', s=20)


    # Añadir etiquetas a los ejes
    plt.xlabel('$α^ℓ$', fontsize=16)
    plt.ylabel('Predicciones', fontsize=16)

    # Aumentar tamaño de los ticks de los ejes
    plt.xticks(fontsize=14)
    plt.yticks(fontsize=14)

    # Ajustar el rango del eje y entre 0 y 1 (o el rango que desees)
    plt.xlim(-0.1, max(alfas) + 0.1)
    plt.ylim(0, 0.55)

    # Guardar el gráfico con un nombre específico para cada algoritmo
    plt.savefig(f"{idx+1}.Logit_{algorithm.split('_')[1]}_FFEF.png", dpi=300)

    # Mostrar el gráfico
    plt.show()

## Predicción: Redes Neuronales

In [ ]:
# Automatizado.
# Crear un DataFrame con un rango de valores de Alpha de 0 a 1 con paso de 0.01
alpha_values = pd.DataFrame({'Alfa': [i / 100 for i in range(101)]})

# Añadir una columna 'const' con valores 1
alpha_values['const'] = 1

# Algoritmos y colores especificados (usando los nombres que el modelo espera)
algorithms = ['Algoritmo_Louvain', 'Algoritmo_Leiden', 'Algoritmo_Walktrap', 
              'Algoritmo_Infomap', 'Algoritmo_Fast Greedy', 'Algoritmo_Surprise']
colors = ['blue', 'green', 'yellow', 'grey', 'purple', 'pink']

# Listas para guardar las predicciones de cada algoritmo
predictions_dict = {}

# Para cada algoritmo, realizar las predicciones y guardar el gráfico
for idx, algorithm in enumerate(algorithms):
    # Establecer el valor de 1 para el algoritmo actual y 0 para los demás
    # Primero restablecemos todos los algoritmos a 0
    alpha_values[algorithms] = 0
    
    # Asignamos 1 al algoritmo actual
    alpha_values[algorithm] = 1

    # Crear el conjunto de características X_test para este algoritmo
    # Usamos feature_names_in_ para asegurarnos del orden correcto
    # Usamos esto para RN:
    X_test_p = alpha_values[X_train.columns]
    
    # Realizar las predicciones con el modelo de Redes neuronales
    y_pred = model_nn.predict(X_test_p)  
    
    # Guardar las predicciones
    predictions_dict[algorithm] = y_pred
    
    # Crear un gráfico de dispersión para este algoritmo
    alfas = alpha_values['Alfa']
    predicciones = y_pred

    # Crear gráfico
    plt.figure(figsize=(8, 6))
    plt.scatter(alfas[:-1], predicciones[:-1], label='Predictions', color=colors[idx], marker='s', s=20)  # Excluye la última predicción
    plt.scatter(alfas.iloc[-1], predicciones[-1], label='Last Prediction', color='red', marker='s', s=20)


    # Añadir etiquetas a los ejes
    plt.xlabel('$α^ℓ$', fontsize=16)
    plt.ylabel('Predicciones', fontsize=16)

    # Aumentar tamaño de los ticks de los ejes
    plt.xticks(fontsize=14)
    plt.yticks(fontsize=14)

    # Ajustar el rango del eje y entre 0 y 1 (o el rango que desees)
    plt.xlim(-0.1, max(alfas) + 0.1)
    plt.ylim(0, 0.55)

    # Guardar el gráfico con un nombre específico para cada algoritmo
    plt.savefig(f"{idx+1}.NeuralNetwork_{algorithm.split('_')[1]}_FFEF.png", dpi=300)

    # Mostrar el gráfico
    plt.show()

## Predicción: Random Forest

In [ ]:
# Automatizado.
# Crear un DataFrame con un rango de valores de Alpha de 0 a 1 con paso de 0.01
alpha_values = pd.DataFrame({'Alfa': [i / 100 for i in range(101)]})

# Añadir una columna 'const' con valores 1
alpha_values['const'] = 1

# Algoritmos y colores especificados (usando los nombres que el modelo espera)
algorithms = ['Algoritmo_Louvain', 'Algoritmo_Leiden', 'Algoritmo_Walktrap', 
              'Algoritmo_Infomap', 'Algoritmo_Fast Greedy', 'Algoritmo_Surprise']
colors = ['blue', 'green', 'yellow', 'grey', 'purple', 'pink']

# Listas para guardar las predicciones de cada algoritmo
predictions_dict = {}

# Para cada algoritmo, realizar las predicciones y guardar el gráfico
for idx, algorithm in enumerate(algorithms):
    # Establecer el valor de 1 para el algoritmo actual y 0 para los demás
    # Primero restablecemos todos los algoritmos a 0
    alpha_values[algorithms] = 0
    
    # Asignamos 1 al algoritmo actual
    alpha_values[algorithm] = 1

    # Crear el conjunto de características X_test para este algoritmo
    # Usamos feature_names_in_ para asegurarnos del orden correcto
    # Usamos esto para RF
    X_test_p = alpha_values[model_rf.feature_names_in_]
    
    # Realizar las predicciones con el modelo de Regresión Logistica
    y_pred = model_rf.predict(X_test_p)
    
    # Guardar las predicciones
    predictions_dict[algorithm] = y_pred
    
    # Crear un gráfico de dispersión para este algoritmo
    alfas = alpha_values['Alfa']
    predicciones = y_pred

    # Crear gráfico
    plt.figure(figsize=(8, 6))
    plt.scatter(alfas[:-1], predicciones[:-1], label='Predictions', color=colors[idx], marker='x', s=20)  # Excluye la última predicción
    plt.scatter(alfas.iloc[-1], predicciones[-1], label='Last Prediction', color='red', marker='x', s=20)


    # Añadir etiquetas a los ejes
    plt.xlabel('$α^ℓ$', fontsize=16)
    plt.ylabel('Predicciones', fontsize=16)

    # Aumentar tamaño de los ticks de los ejes
    plt.xticks(fontsize=14)
    plt.yticks(fontsize=14)

    # Ajustar el rango del eje y entre 0 y 1 (o el rango que desees)
    plt.xlim(-0.1, max(alfas) + 0.1)
    plt.ylim(0, 0.55)

    # Guardar el gráfico con un nombre específico para cada algoritmo
    plt.savefig(f"{idx+1}.RandomForest_{algorithm.split('_')[1]}_FFEF.png", dpi=300)

    # Mostrar el gráfico
    plt.show()

## Predicción: XGBoost

In [ ]:
# Automatizado.
# Crear un DataFrame con un rango de valores de Alpha de 0 a 1 con paso de 0.01
alpha_values = pd.DataFrame({'Alfa': [i / 100 for i in range(101)]})

# Añadir una columna 'const' con valores 1
alpha_values['const'] = 1

# Algoritmos y colores especificados (usando los nombres que el modelo espera)
algorithms = ['Algoritmo_Louvain', 'Algoritmo_Leiden', 'Algoritmo_Walktrap', 
              'Algoritmo_Infomap', 'Algoritmo_Fast Greedy', 'Algoritmo_Surprise']
colors = ['blue', 'green', 'yellow', 'grey', 'purple', 'pink']

# Listas para guardar las predicciones de cada algoritmo
predictions_dict = {}

# Para cada algoritmo, realizar las predicciones y guardar el gráfico
for idx, algorithm in enumerate(algorithms):
    # Establecer el valor de 1 para el algoritmo actual y 0 para los demás
    # Primero restablecemos todos los algoritmos a 0
    alpha_values[algorithms] = 0
    
    # Asignamos 1 al algoritmo actual
    alpha_values[algorithm] = 1

    # Crear el conjunto de características X_test para este algoritmo
    # Usamos feature_names_in_ para asegurarnos del orden correcto
    # Usamos esto para RF
    X_test_p = alpha_values[model_xgb.feature_names_in_]
    
    # Realizar las predicciones con el modelo de Regresión Logistica
    y_pred = model_xgb.predict(X_test_p)
    
    # Guardar las predicciones
    predictions_dict[algorithm] = y_pred
    
    # Crear un gráfico de dispersión para este algoritmo
    alfas = alpha_values['Alfa']
    predicciones = y_pred

    # Crear gráfico
    plt.figure(figsize=(8, 6))
    plt.scatter(alfas[:-1], predicciones[:-1], label='Predictions', color=colors[idx], marker='D', s=20)  # Excluye la última predicción
    plt.scatter(alfas.iloc[-1], predicciones[-1], label='Last Prediction', color='red', marker='D', s=20)


    # Añadir etiquetas a los ejes
    plt.xlabel('$α^ℓ$', fontsize=16)
    plt.ylabel('Predicciones', fontsize=16)

    # Aumentar tamaño de los ticks de los ejes
    plt.xticks(fontsize=14)
    plt.yticks(fontsize=14)

    # Ajustar el rango del eje y entre 0 y 1 (o el rango que desees)
    plt.xlim(-0.1, max(alfas) + 0.1)
    plt.ylim(0, 0.55)

    # Guardar el gráfico con un nombre específico para cada algoritmo
    plt.savefig(f"{idx+1}.XGBoost_{algorithm.split('_')[1]}_FFEF.png", dpi=300)

    # Mostrar el gráfico
    plt.show()